# 01 · Rebuild the fly feeding benchmark

**Project question**  
Do revisions to the wiring map change which cell types a fly-brain model predicts will trigger a feeding movement?

**First checkpoint**  
Rebuild one published result from its 106 underlying rows. You will find out which experimental responses the model missed and how two very small signals affect the score. This notebook analyses firing rates the authors already computed on FlyWire v630. A fresh neural simulation is a later checkpoint.

Run the cells in order. In Google Colab, use **Runtime → Run all** for the first pass. Then go to **Step 2**, change **left** to **either**, and rerun that cell. Steps 1–3 have the data embedded and work offline with Python, pandas and IPython. Step 4 optionally checks the original files online.

## Step 1 · Understand one row

Each row is a **cell type**. In the simulation the researchers stimulated those cells and recorded the firing of the left and right **MN9** motor neurons. MN9 controls extension of part of the fly’s feeding tube. In a separate laboratory experiment they activated the same cell type in living flies and recorded what fraction extended that part. Where several experimental lines tested a cell type, Figure 2 uses the line with the highest response.

Our primary comparison uses **50 Hz** stimulation. A positive experimental outcome means the recorded fraction is greater than zero. The 200 Hz columns are for a separate check of how the two model outputs were scored. [Shiu and colleagues, Figure 2 and Supplementary Table 3](https://pmc.ncbi.nlm.nih.gov/articles/PMC11446845/).

In [36]:
# @title Load the published data and inspect three examples
from io import StringIO
from pathlib import Path

import pandas as pd
from IPython.display import display

# Numeric extract from Supplementary Table 3. Step 4 can verify every value.
source_csv = """cell_type,observed_fraction,left_50,right_50,left_200,right_200
roundup,1,79.5,82.6666666666666,154.7,157.7
diatom,0.6,29.1333333333333,28.3333333333333,89.5,107.766666666666
sink_sync,0.2,22.1666666666666,22.7666666666666,84.0666666666666,87.9333333333333
G2N_1,1,12.3666666666666,13.9666666666666,80.2,85.1333333333333
clavicle,0.5,10.5,7.1,77.5666666666666,73.5333333333333
Fdg,0.8,22.5333333333333,20.3333333333333,74.7666666666666,65.4666666666666
bract,0.8,33.6666666666666,25.2333333333333,63.9666666666666,56.0666666666666
vice,0,0,0,39.9333333333333,47.1333333333333
rattle,0.9,1.86666666666666,1.16666666666666,39.6,29.5666666666666
FMIn,1,0.366666666666666,0.1,32.2666666666666,33.6
TH_VUM,1,0.0333333333333333,2.46666666666666,13.7,34.4333333333333
aster,0,0,0,12.0666666666666,14.1666666666666
seagull,0,0,0,6.46666666666666,0.9
mist,0,0,0,3.83333333333333,0
kitty,0,7.3,10.8666666666666,3.13333333333333,5.63333333333333
fudog,0,0,0,1.33333333333333,2.33333333333333
aDT6,0,0,0,0,0
aSG1,0,0,0,0,0
aSG7,0,0,0,0,0
amulet,0,0,0,0,0
asteroid,0,0,0,0,0
bamboo,0,0,0,0,0
basket,0,0,0,0,0
bluebell,0,0,0,0,0
bobber,0,0,0,0,0
box,0,0,0,0,0
bridle,0,0,0,0,0
brontosaraus,0,0,0,0,0
broom,0,0,0,0,0
buddy,0,0,0,0,0
coy,0,0,0,0,0
crab,0,0,0,0,0
cradle,0,0,0,0,0
damsel,0,0,0,0,0
doublescoop,0,0,0,0,0
earmuff,0,0,0,0,0
eiffel,0,0,0,0,0
fluff,0,0,0,0,0
foxglove,0,0,0,0,0
gallinule,0,0,0,0,0
gazebo,0,0,0,0,0
genie,0,0,0,0,0
gnome,0,0,0,0,0
good_dog,0,0,0,0,0
gumdrop,0,0,0,0,0
handle,0,0,0,0,0
handup,0,0,0,0,0
haystack,0,0,0,0,0
horn,0,0,0,0,0
horntail,0,0,0,0,0
horseshoe,0,0,0,0,0
hound,0,0,0,0,0
hyacinth,0,0,0,0,0
justice,0,0,0,0,0
kelp,0,0,0,0,0
knees,0,0,0,0,0
kokopelli,0,0,0,0,0
landslide,0,0,0,0,0
lion,0,0,0,0,0
mandala,0,0,0,0,0
marge,0.1,0,0,0,0
meteor,0,0,0,0,0
mime,0,0,0,0,0
moor,0,0,0,0,0
mothership,0,0,0,0,0
mute,0,0,0,0,0
nagini,0,0,0,0,0
nori,0,0,0,0,0
oink,0,0,0,0,0
OinkT,0,0,0,0,0
oval,0,0,0,0,0
pSG1,0,0,0,0,0
peacock,0,0,0.0333333333333333,0,0.0666666666666666
peafowl,0,0,0,0,0
peahen,0,0,0.0333333333333333,0,0.2
phantom,0.6,0,0,0,0
planter,0,0,0,0,0
pleco,0,0,0,0,0
pringle,0,0,0,0,0
puddle,0,0,0,0,0
rocket,0,0,0,0,0
rose,0,0,0,0,0
ruby,0,0,0,0,0
Salivary_MN13,0,0,0,0,0
shark,0,0,0,0,0
snake,0,0,0,0,0
spirit,0,0,0,0,0
spray,0,0,0,0,0
sullivan,0,0,0,0,0.0666666666666666
sundrop,0,0,0,0,0
tentacular,0.5,0,0,0,0
tinctoria,0,0,0,0,0
tophat,0,0,0,0,0
TPN4,0,0,0,0,0.1
trident,0,0,0,0,0
trogon,0,0,0,0,0
trumpet,0,0,0,0,0
tulip,0,0,0,0,0
tundra,0,0,0,0,0
turner,0,0,0,0,0
twirl,0,0,0,0,0
usnea,1,0,0,0,0
wafflecone,0,0,0,0,0
weaver,0,0,0,0,0
web,0,0,0,0,0
whisker,0,0,0,0,0
"""
screen = pd.read_csv(StringIO(source_csv))
assert len(screen) == 106 and screen["cell_type"].is_unique
assert screen["observed_fraction"].between(0, 1).all()
rate_columns = ["left_50", "right_50", "left_200", "right_200"]
assert screen[rate_columns].notna().all().all() and (screen[rate_columns] >= 0).all().all()

observed = screen["observed_fraction"] > 0
print(f"{len(screen)} cell types. {int(observed.sum())} produced a movement in the lab.")
examples = screen["cell_type"].isin(["roundup", "kitty", "usnea"])
display(screen.loc[examples, ["cell_type", "observed_fraction", "left_50", "right_50"]])

106 cell types. 14 produced a movement in the lab.


,cell_type,observed_fraction,left_50,right_50
0,roundup,1.0,79.5,82.666667
14,kitty,0.0,7.3,10.866667
101,usnea,1.0,0.0,0.000000


Read **roundup** first. Its observed fraction is 1, and the model’s left MN9 fired at 79.5 Hz. Now look at **usnea**. Its observed fraction is also 1, yet both model neurons have a firing rate of zero. These two rows give you one correct prediction and one missed response.

## Step 2 · Score the model

The paper counts a non-zero MN9 firing rate as a predicted movement. The spreadsheet has two MN9 columns. Begin with the one labelled **left** and read the four counts. Then change the setting to **either** and run the same cell again. Before doing so, consider whether allowing a second neuron to trigger a positive prediction can ever reduce the number of positive predictions.

In [37]:
readout = "either"  # @param ["left", "right", "either", "both"]

def predicted_movement(frame, hz, rule):
    left_fired = frame[f"left_{hz}"] > 0
    right_fired = frame[f"right_{hz}"] > 0
    return {
        "left": left_fired,
        "right": right_fired,
        "either": left_fired | right_fired,
        "both": left_fired & right_fired,
    }[rule]

def outcome_counts(predicted, observed):
    return {
        "true_positive": int((predicted & observed).sum()),
        "false_positive": int((predicted & ~observed).sum()),
        "false_negative": int((~predicted & observed).sum()),
        "true_negative": int((~predicted & ~observed).sum()),
    }

predicted = predicted_movement(screen, 50, readout)
counts = outcome_counts(predicted, observed)
matrix = pd.DataFrame(
    [[counts["true_positive"], counts["false_positive"]],
     [counts["false_negative"], counts["true_negative"]]],
    index=["Predicted movement", "Predicted no movement"],
    columns=["Observed movement", "Observed no movement"],
)
display(matrix)
correct = counts["true_positive"] + counts["true_negative"]
print(f"{readout} readout at 50 Hz → {correct}/{len(screen)} correct ({correct / len(screen):.1%}).")
print(f"Detected {counts['true_positive']} of {int(observed.sum())} cell types with observed movement.")
print("False positives —", ", ".join(screen.loc[predicted & ~observed, "cell_type"]))
print("Missed responses —", ", ".join(screen.loc[~predicted & observed, "cell_type"]))

,Observed movement,Observed no movement
Predicted movement,10,3
Predicted no movement,4,89


either readout at 50 Hz → 99/106 correct (93.4%).
Detected 10 of 14 cell types with observed movement.
False positives — kitty, peacock, peahen
Missed responses — marge, phantom, tentacular, usnea


## Step 3 · Find the cells behind the change

Compare **left** with **either** while holding the 106 rows, the stimulus and the observed outcomes fixed. The next cell displays every row whose prediction changes, checks the paper’s additional result at 200 Hz, and saves a small CSV with the classifications from both 50 Hz rules.

In [38]:
left_50 = predicted_movement(screen, 50, "left")
either_50 = predicted_movement(screen, 50, "either")
columns = ["cell_type", "observed_fraction", "left_50", "right_50"]
display(screen.loc[left_50 != either_50, columns])

rows = []
for rule in ["left", "right", "either", "both"]:
    at_50 = outcome_counts(predicted_movement(screen, 50, rule), observed)
    at_200 = outcome_counts(predicted_movement(screen, 200, rule), observed)
    rows.append({
        "readout": rule,
        "correct at 50 Hz": at_50["true_positive"] + at_50["true_negative"],
        "false positives at 50 Hz": at_50["false_positive"],
        "false positives at 200 Hz": at_200["false_positive"],
        "increase in false positives": at_200["false_positive"] - at_50["false_positive"],
    })
display(pd.DataFrame(rows).set_index("readout"))

export = screen.assign(observed_movement=observed, predicted_left=left_50, predicted_either=either_50)
output_file = Path("fly_feeding_50hz_predictions.csv")
export.to_csv(output_file, index=False)
print(f"Saved {len(export)} rows to {output_file.name}.")

,cell_type,observed_fraction,left_50,right_50
72,peacock,0.0,0.0,0.033333
74,peahen,0.0,0.0,0.033333


,correct at 50 Hz,false positives at 50 Hz,false positives at 200 Hz,increase in false positives
readout,,,,
left,101,1,6,5
right,99,3,9,6
either,99,3,10,7
both,101,1,5,4


Saved 106 rows to fly_feeding_50hz_predictions.csv.


**What the check establishes.** At 50 Hz, both the left-only rule and the rule requiring both sides give the published counts. The paper also says that raising stimulation from 50 to 200 Hz added five false positives. Among the four rules above, only the left-column rule matches the counts at 50 Hz and that additional observation. We use **left** as the working primary rule and retain **either** as a sensitivity check. This is an inference about scoring from the published numbers. It does not establish that only the biological left neuron matters.

**Write one sentence in your own words.** What changed for *peacock* and *peahen*, and why does it matter when the classification threshold is greater than zero? You can save your edited notebook and the CSV as the first project checkpoint.

## Step 4 · Verify the sources (optional)

The next cell retrieves the original Excel workbook from the publisher or the PubMed Central archive, or reads `41586_2024_7763_MOESM2_ESM.xlsx` in the current folder if you already have it. It checks the workbook fingerprint and compares this notebook’s extract with the 106 original rows. The mean-rate columns precede a second block of similarly named standard-deviation columns. Set `verify_sources` to `False` below to skip the source checks.

The supplementary numeric data are credited to Shiu and colleagues (2024). The extract selects and renames six columns. [Original article and supplementary materials](https://pmc.ncbi.nlm.nih.gov/articles/PMC11446845/) · [Original workbook](https://pmc-oa-opendata.s3.amazonaws.com/PMC11446845.1/41586_2024_7763_MOESM2_ESM.xlsx) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [39]:
verify_sources = True  # @param {type:"boolean"}

import hashlib
from io import BytesIO
from urllib.request import urlopen
from zipfile import is_zipfile

def download(url):
    with urlopen(url, timeout=30) as response:
        return response.read()

def check_fingerprint(content, expected):
    actual = hashlib.sha256(content).hexdigest()
    if actual != expected:
        raise ValueError(f"Source fingerprint changed. Expected {expected}, received {actual}.")

def verify_workbook():
    import openpyxl

    local_book = Path("41586_2024_7763_MOESM2_ESM.xlsx")
    if local_book.is_file():
        book_bytes = local_book.read_bytes()
    else:
        mirrors = [
            ("https://media.springernature.com/full/springer-static/esm/"
             "art%3A10.1038%2Fs41586-024-07763-9/MediaObjects/"),
            "https://pmc-oa-opendata.s3.amazonaws.com/PMC11446845.1/",
        ]
        for base_url in mirrors:
            try:
                candidate = download(base_url + local_book.name)
            except OSError:
                continue
            if is_zipfile(BytesIO(candidate)):
                book_bytes = candidate
                break
        else:
            raise ConnectionError("No source returned an Excel workbook.")
    check_fingerprint(book_bytes, "6922e16825aa0c92a28e2a634b073dcad7641e9c9283c4b9d8ee1e34e6d9b8d9")
    book = openpyxl.load_workbook(BytesIO(book_bytes), read_only=True, data_only=True)
    try:
        sheet = book["Sup Table 3 Predicted MN9 vs. o"]
        rows = sheet.values
        header = next(rows)
        positions = [0, 1, 12, 13, 18, 19]  # Excel columns A, B, M, N, S, T
        expected_header = [None, "MN9 Optogenetic Activation Rate", "50 Hz MN9_Left",
                           "50 Hz MN9_Right", "200 Hz MN9_Left", "200 Hz MN9_Right"]
        assert [header[i] for i in positions] == expected_header
        data = [[row[i] for i in positions] for row in rows if row[0] is not None]
    finally:
        book.close()
    original = pd.DataFrame(data, columns=screen.columns)
    pd.testing.assert_frame_equal(original, screen, check_dtype=False, rtol=1e-12, atol=1e-15)
    print("Verified all 106 rows and all six columns against Supplementary Table 3.")

if verify_sources:
    try:
        verify_workbook()
    except OSError as error:
        print("Source download unavailable. Steps 1–3 used the embedded extract.", error)
else:
    print("Using the embedded extract. Online check skipped.")

Verified all 106 rows and all six columns against Supplementary Table 3.


### A first check for the next experiment

A **root ID** identifies a reconstructed neuron in a particular map release. A cell type can contain several neurons. Blindly reusing its old list can silently change the input we stimulate. This optional cell checks the authors’ 106 input groups against the two released neuron lists. An unchanged ID is a candidate match. A missing ID calls for checking reconstruction history. Neither outcome alone proves anatomical equivalence.

The files are pinned to the authors’ repository at commit `91bdd1e`. [Original code and datasets](https://github.com/philshiu/Drosophila_brain_model/tree/91bdd1e7dcf193f3e7ca5a8933497fcef63b7960) · [FlyWire’s note on changing IDs](https://codex.flywire.ai/faq).

In [40]:
def examine_input_ids():
    import csv
    import pickle

    repo = ("https://raw.githubusercontent.com/philshiu/Drosophila_brain_model/"
            "91bdd1e7dcf193f3e7ca5a8933497fcef63b7960/")
    fingerprints = {
        "sez_neurons.pickle": "2bc5a2a8289924f2de39a8e68bdc07025cb6d6c6c6ddc83a973c0cace7cec2ce",
        "2023_03_23_completeness_630_final.csv": "e6b71e17671a9bdb05f55e4bc6774640a1418cb7a05125e0fc994ad40f9bfdfb",
        "Completeness_783.csv": "bbb847a4cc2caaa7a16349722d220c087317b946d148d4d592d94d250617a311",
    }

    def pinned_file(name):
        data = download(repo + name)
        check_fingerprint(data, fingerprints[name])
        return data

    def read_ids(name):
        rows = csv.reader(StringIO(pinned_file(name).decode("utf-8")))
        next(rows)
        return {int(row[0]) for row in rows}

    class SimpleDataReader(pickle.Unpickler):
        def find_class(self, module, name):
            raise ValueError("The reference file should contain plain lists of neuron IDs.")

    groups = SimpleDataReader(BytesIO(pinned_file("sez_neurons.pickle"))).load()
    old = read_ids("2023_03_23_completeness_630_final.csv")
    new = read_ids("Completeness_783.csv")
    assert len(groups) == len(screen)
    assert {name.casefold() for name in groups} == {name.casefold() for name in screen.cell_type}
    rows = []
    for name, members in groups.items():
        members = set(members)
        assert members and members <= old
        retained = members & new
        if retained == members:
            status = "all old IDs present"
        elif retained:
            status = "some old IDs present"
        else:
            status = "no old IDs present"
        rows.append({"cell_type": name, "v630_input_ids": len(members),
                     "same_ids_in_v783": len(retained), "status": status})
    inventory = pd.DataFrame(rows).sort_values("cell_type")
    display(inventory["status"].value_counts().rename("cell types").to_frame())
    inventory.to_csv("fly_feeding_input_id_inventory.csv", index=False)
    mn9_ids = {"left (as labelled by authors)": 720575940660219265,
               "right (as labelled by authors)": 720575940645521262}
    for label, root in mn9_ids.items():
        print(f"MN9 {label}: old root present in v630 = {root in old}, in v783 = {root in new}.")

if verify_sources:
    try:
        examine_input_ids()
    except OSError as error:
        print("Input ID files unavailable. The screening-table results above are complete.", error)
else:
    print("Online input ID inventory skipped.")

,cell types
status,
all old IDs present,63
some old IDs present,38
no old IDs present,5


MN9 left (as labelled by authors): old root present in v630 = True, in v783 = True.
MN9 right (as labelled by authors): old root present in v630 = True, in v783 = False.


## Next checkpoint

Run a small v630 simulation using the authors’ code and the same primary MN9 output. Start with a clear positive and a well-characterised mismatch such as *roundup* and *usnea*. Repeat the original-map runs to quantify run-to-run variation before interpreting a cross-map difference. The second release and any changed cell identifiers need explicit matching before the final comparison.

The results in this notebook are a reanalysis of published data and a check of identifier coverage. The full v630 simulation and the v630–v783 behavioural comparison are still to come.